# Phase 3 — Feature Engineering

Every feature is defined once, in `src/features.py`, and imported by both the
training pipeline and the live API. That is deliberate: reimplementing the same
logic in a backend is the most common way a churn model quietly degrades after
deployment.

In [1]:
import sys, sqlite3, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd, numpy as np
pd.set_option('display.width', 120)

from config import DB_PATH
from features import engineer_features, FEATURE_JUSTIFICATIONS
df = pd.read_sql_query('SELECT * FROM v_customer_360', sqlite3.connect(DB_PATH))
feats = engineer_features(df)
print(f'{df.shape[1]} raw -> {feats.shape[1]} engineered columns')
feats.head(3)

22 raw -> 30 engineered columns


,tenure,monthly_charges,total_charges,total_services,protection_services,avg_monthly_spend_ratio,charge_trend_delta,contract_ord,tenure_group_ord,senior_citizen,...,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,payment_method,contract,tenure_group
0,1,29.85,29.85,1,1,29.8500,0.0000,0,0,0,...,DSL,No,Yes,No,No,No,No,Electronic check,Month-to-month,0-12
1,34,56.95,1889.50,2,2,55.5735,1.3765,1,2,0,...,DSL,Yes,No,Yes,No,No,No,Mailed check,One year,25-48
2,2,53.85,108.15,2,2,54.0750,-0.2250,0,0,0,...,DSL,Yes,Yes,No,No,No,No,Mailed check,Month-to-month,0-12


## Justification — one entry per engineered feature

In [2]:
for name, why in FEATURE_JUSTIFICATIONS:
    print(f'* {name}\n    {why}\n')

* total_services
    Count of the six optional add-ons. Each additional product is a switching cost. EDA Q10 shows churn falling from 45.8% (one add-on) to 5.3% (six).

* protection_services
    Security/backup/device-protection/tech-support only, excluding streaming. Support-shaped products correlate with retention far more strongly than entertainment add-ons (41.6% vs 15.2% churn on tech support alone).

* avg_monthly_spend_ratio
    TotalCharges / tenure — historical average spend. Separates a customer who has always paid $90 from one recently upgraded to $90. Guarded for tenure=0, where the current plan price is substituted.

* charge_trend_delta
    monthly_charges minus the historical average. Positive values flag a recent upsell or price rise, which is what triggers a cancellation call.

* contract_ord
    Ordinal 0/1/2 for month-to-month/one-year/two-year. Contract length is genuinely ordered, so one-hot encoding would discard the ordering that drives the strongest single effec

## Does each engineered feature actually separate the classes?

In [3]:
check = feats.copy(); check['churn'] = df.churn_flag.values
rows = []
for col in ['total_services', 'protection_services', 'charge_trend_delta',
            'new_customer_risk_flag', 'manual_payment_flag', 'no_protection_flag',
            'is_fiber', 'contract_ord', 'tenure_group_ord']:
    rows.append({'feature': col,
                 'churned_mean': round(check.loc[check.churn == 1, col].mean(), 3),
                 'retained_mean': round(check.loc[check.churn == 0, col].mean(), 3),
                 'r_with_churn': round(check[col].corr(check.churn), 3)})
pd.DataFrame(rows).sort_values('r_with_churn', key=abs, ascending=False)

,feature,churned_mean,retained_mean,r_with_churn
7,contract_ord,0.140,0.889,-0.397
3,new_customer_risk_flag,0.323,0.058,0.349
8,tenure_group_ord,0.847,1.806,-0.345
5,no_protection_flag,0.384,0.106,0.320
6,is_fiber,0.694,0.348,0.308
4,manual_payment_flag,0.738,0.502,0.210
1,protection_services,0.895,1.399,-0.173
0,total_services,1.768,2.135,-0.088
2,charge_trend_delta,0.008,-0.005,0.002


## The risk flag, checked against the segment it was built from

In [4]:
flagged = check[check.new_customer_risk_flag == 1]
print(f'flagged customers : {len(flagged):,}')
print(f'churn when flagged: {flagged.churn.mean():.1%}')
print(f'churn otherwise   : {check[check.new_customer_risk_flag==0].churn.mean():.1%}')

flagged customers : 904
churn when flagged: 66.7%
churn otherwise   : 20.6%


## Leakage check

Every feature is computable **before** a customer cancels: demographics, contract
terms, subscribed products, billing to date. Nothing reads `churn`, and there is no
target encoding or churn-rate-by-segment feature — those would leak the label
through an aggregate and produce a model that looks excellent in validation and
fails on the first customer it has never seen.

In [5]:
assert 'churn' not in feats.columns and 'churn_flag' not in feats.columns
print('no target column present in the feature matrix')
print('nulls after engineering:', int(feats.isna().sum().sum()))

no target column present in the feature matrix
nulls after engineering: 0
